In [7]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import math
from tokenizers import Tokenizer
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [8]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/punitkashyap2007/virgo-base-model/virgo_base_final.pt
/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json


In [9]:
CHECKPOINT_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-base-model/virgo_base_final.pt"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

print("✅ Checkpoint Loaded Successfully")
print()
print(checkpoint.keys())

✅ Checkpoint Loaded Successfully

dict_keys(['epoch', 'step', 'tokens_seen', 'model_state_dict', 'optimizer_state_dict', 'best_val_loss', 'cumulative_correct', 'cumulative_total', 'config'])


In [10]:
TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

In [11]:
config = checkpoint["config"]

print("✅ Model Configuration")
print("-" * 40)

for key, value in config.items():
    print(f"{key:20}: {value}")

✅ Model Configuration
----------------------------------------
vocab_size          : 45000
d_model             : 768
num_heads           : 12
num_layers          : 12
d_ff                : 3072
max_seq_length      : 1024
dropout             : 0.1


In [12]:
class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, max_seq_length=2048, base=10000.0):
        super(RotaryEmbedding, self).__init__()

        assert d_k % 2 == 0, "Head dimension must be even for RoPE"

        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        positions = torch.arange(max_seq_length).float()

        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, Q, K):
        seq_length = Q.size(-2)

        cos = self.cos[:, :, :seq_length, :].to(dtype=Q.dtype)
        sin = self.sin[:, :, :seq_length, :].to(dtype=Q.dtype)

        Q_even = Q[..., 0::2]
        Q_odd = Q[..., 1::2]

        K_even = K[..., 0::2]
        K_odd = K[..., 1::2]

        Q = torch.stack((Q_even * cos - Q_odd * sin, Q_even * sin + Q_odd * cos), dim=-1).flatten(-2)
        K = torch.stack((K_even * cos - K_odd * sin, K_even * sin + K_odd * cos), dim=-1).flatten(-2)

        return Q, K

In [13]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

In [14]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_length):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.rope = RotaryEmbedding(self.d_k, max_seq_length)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        Q, K = self.rope(Q, K)

        attn_output = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.0, is_causal=True)

        return self.W_o(self.combine_heads(attn_output))

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, max_seq_length)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        norm_x = self.norm1(x)
        x = x + self.dropout(self.self_attn(norm_x))

        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))

        return x

In [16]:
class VirgoBase(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(VirgoBase, self).__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.apply(self._init_weights)

        self.lm_head.weight = self.token_embedding.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids):
        seq_length = input_ids.size(1)

        if seq_length > self.max_seq_length:
            raise ValueError(f"Sequence length {seq_length} exceeds maximum {self.max_seq_length}")

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        return self.lm_head(x)

In [17]:
model = VirgoBase(
    vocab_size=config["vocab_size"],
    d_model=config["d_model"],
    num_heads=config["num_heads"],
    num_layers=config["num_layers"],
    d_ff=config["d_ff"],
    max_seq_length=config["max_seq_length"],
    dropout=0.0
)

In [18]:
model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

print("✅ Virgo Base loaded successfully!")

✅ Virgo Base loaded successfully!


In [19]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("✅ Tokenizer loaded successfully!")
print("Vocabulary Size:", tokenizer.get_vocab_size())

✅ Tokenizer loaded successfully!
Vocabulary Size: 45000


In [20]:
max_seq_length = 1024

In [21]:
import sys
import time
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate_virgo(
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
):

    model.eval()

    # Encode prompt
    encoding = tokenizer.encode(prompt)
    ids = encoding.ids

    # Prevent empty prompt
    if len(ids) == 0:
        bos_id = tokenizer.token_to_id("<bos>")
        if bos_id is None:
            raise ValueError("Prompt produced zero tokens and tokenizer has no <bos> token.")
        ids = [bos_id]

    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    generated = input_ids.clone()

    eos_id = tokenizer.token_to_id("<eos>")

    for _ in range(max_new_tokens):

        # Sliding context
        context = generated[:, -config["max_seq_length"]:]

        logits = model(context)

        next_token_logits = logits[:, -1, :].float()

        # Repetition penalty
        if repetition_penalty != 1.0:

            previous_tokens = torch.unique(generated)

            next_token_logits[:, previous_tokens] = torch.where(
                next_token_logits[:, previous_tokens] < 0,
                next_token_logits[:, previous_tokens] * repetition_penalty,
                next_token_logits[:, previous_tokens] / repetition_penalty,
            )

        # Greedy decoding
        if temperature == 0:

            next_token = torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True
            )

        else:

            next_token_logits /= temperature

            # Top-k
            if top_k is not None:

                k = min(top_k, next_token_logits.size(-1))

                values, _ = torch.topk(next_token_logits, k)

                next_token_logits[
                    next_token_logits < values[:, [-1]]
                ] = -float("inf")

            # Top-p
            if top_p is not None:

                sorted_logits, sorted_indices = torch.sort(
                    next_token_logits,
                    descending=True
                )

                probs = F.softmax(sorted_logits, dim=-1)

                cumulative = torch.cumsum(probs, dim=-1)

                remove = cumulative > top_p
                remove[:, 1:] = remove[:, :-1].clone()
                remove[:, 0] = False

                sorted_logits[remove] = -float("inf")

                next_token_logits = torch.full_like(
                    next_token_logits,
                    -float("inf")
                )

                next_token_logits.scatter_(
                    1,
                    sorted_indices,
                    sorted_logits
                )

            probs = F.softmax(next_token_logits, dim=-1)

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

        generated = torch.cat(
            (generated, next_token),
            dim=1
        )

        if eos_id is not None and next_token.item() == eos_id:
            break

    prompt_length = input_ids.size(1)

    return tokenizer.decode(
        generated[0][prompt_length:].tolist(),
        skip_special_tokens=True
    )

In [26]:
import sys
import time


def type_writer(text, delay=0.01):

    for ch in text:
        print(ch, end="", flush=True)
        time.sleep(delay)

    print()


def virgo_chat(
    prompt,
    max_new_tokens=256,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    typing_delay=0.003
):

    start_time = time.time()

    output = generate_virgo(
        prompt=prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )

    elapsed = time.time() - start_time

    print()
    print("╭────────────────────────────────────────────────────────────────────╮")
    print("│                             VIRGO BASE                             │")
    print("╰────────────────────────────────────────────────────────────────────╯")

    print()
    print("◆ PROMPT")
    print("──────────────────────────────────────────────────────────────────────")
    print(prompt)

    print()
    print("◆ VIRGO")
    print("──────────────────────────────────────────────────────────────────────")

    type_writer(output, typing_delay)

    print()
    print("──────────────────────────────────────────────────────────────────────")
    print(f"Generated in {elapsed:.2f}s")
    print("──────────────────────────────────────────────────────────────────────")

    # return output

In [48]:
prompt = input("Enter the prompt: ")

virgo_chat(
    prompt=prompt,
    # max_new_tokens=256,
    temperature=0.8,
    top_p=0.95
    
)


Enter the prompt:  enter the prompt which 



╭────────────────────────────────────────────────────────────────────╮
│                             VIRGO BASE                             │
╰────────────────────────────────────────────────────────────────────╯

◆ PROMPT
──────────────────────────────────────────────────────────────────────
enter the prompt which 

◆ VIRGO
──────────────────────────────────────────────────────────────────────
 indicates that the state of a finite class $H$ in terms of an $s$-dimensional  complex field is dual to a $q$-dimension of $\mathbbR}^d$, and is hence  self-adjoint. We show that there are several classes of this class of theories  with real (i.e., not necessarily non-vanishing) superconformal invariants. We also  study the effects of the Chern-Simons term on the topology of these theories and  discuss the relation between the two theories. The $q$-dimensional theory is  self-adjoint under the action of a single scalar. The $q$-dimensional theory  admits a dual model. We obtain an explicit rel